# Quando a chuva vira problema

**Chuva e ocorrências urbanas nos bairros do Rio de Janeiro**

Pedro Henrique Vasconcelos Neves e Gabriel Andrade Souza · Análise Exploratória de Dados e Visualização · FGV EMAp 2026.2

A ideia: quanta chuva, e em que lugar, vira problema no Rio? E por que a mesma chuva causa estragos diferentes em bairros diferentes?

## 1. De onde vêm os dados

| Base | Fonte | O que tem |
|---|---|---|
| Chuva | Pluviômetros do [Sistema Alerta Rio](https://www.sistema-alerta-rio.com.br/download/dados-meteorologicos/) | Acumulados de 15 min a 96 h, a cada 15 min, desde 1997 |
| Ocorrências | [Centro de Operações Rio (COR)](https://cor.rio/historia/), [publicadas no datalake](https://cor.rio/prefeitura-do-rio-lanca-dados-do-cor-em-datalake-disponivel-ao-cidadao/) | Cada evento registrado: tipo, gravidade, local, início e fim, órgãos acionados |
| Bairros | Limites de bairro do IPP (datalake da prefeitura) | Polígono de cada bairro |
| Censo 2022 | [Censo 2022 por bairro](https://www.data.rio/datasets/fd354740f1934bf5bf8e9b0e2b509aa9_2/about) (IBGE, agregado pelo IPP) | População e domicílios |
| IPS 2022 | [IPS Bairros 2022](https://ips-rio-pcrj.hub.arcgis.com/pages/ipsbairros) | Índice de Progresso Social e indicadores como população em favelas não urbanizadas e domicílios inadequados |

Chuva, ocorrências e bairros estão no **datalake público da prefeitura** (BigQuery, projeto `datario`). Censo e IPS estão no **Data.Rio**.

## 2. Baixando

O datalake é público: basta uma conta Google e uma consulta SQL. Por exemplo:

```python
pandas_gbq.read_gbq("SELECT * FROM `datario.adm_cor_comando.ocorrencias`", project_id="<seu projeto>")
```

Para não repetir o download, guardamos tudo em `dados/brutos/` com dois scripts (cerca de 1,5 GB):

In [ ]:
# Já rodado; descomentar para baixar de novo
# !python scripts/baixar_datario.py
# !python scripts/baixar_data_rio.py

## 3. Uma olhada nos dados

In [ ]:
from datetime import date
from pathlib import Path

import geopandas as gpd
import pandas as pd

pd.set_option("display.max_columns", None)

BQ = Path("dados/brutos/bigquery")
DATA_RIO = Path("dados/brutos/data_rio")

### Ocorrências do COR

In [ ]:
# Um evento por linha, de 2015 a abril de 2024
ocorrencias = pd.read_parquet(BQ / "adm_cor_comando" / "ocorrencias.parquet")
print(ocorrencias.shape)
ocorrencias.head()

In [ ]:
# O tipo de evento vem como código (id_pop); outra tabela dá o nome de cada código
pop = pd.read_parquet(BQ / "adm_cor_comando" / "procedimento_operacional_padrao.parquet")
ocorrencias = ocorrencias.merge(pop, on="id_pop", how="left")
ocorrencias["pop_titulo"].value_counts().head(10)

### Chuva

In [ ]:
# Os pluviômetros do Alerta Rio: onde ficam e se estão operando
estacoes = pd.read_parquet(BQ / "clima_pluviometro" / "estacoes_alertario.parquet")
print(estacoes.shape)
estacoes.head()

In [ ]:
# A série completa tem ~29 milhões de leituras; aqui lemos só um dia: o temporal de 8 de abril de 2019
chuva = pd.read_parquet(
    BQ / "clima_pluviometro" / "taxa_precipitacao_alertario.parquet",
    filters=[("data_particao", "==", date(2019, 4, 8))],
)
print(chuva.shape)
chuva.head()

In [ ]:
# Maior acumulado em 24 h em cada estação nesse dia (mm)
(
    chuva.groupby("id_estacao")["acumulado_chuva_24_h"].max()
    .rename("max_24h_mm")
    .reset_index()
    .merge(estacoes[["id_estacao", "estacao"]], on="id_estacao")
    .sort_values("max_24h_mm", ascending=False)
    .head(10)
)

### O que a cidade registrou nesse temporal

In [ ]:
# Ocorrências abertas em 8 e 9 de abril de 2019, por tipo
temporal = ocorrencias[ocorrencias["data_inicio"].dt.date.isin([date(2019, 4, 8), date(2019, 4, 9)])]
print(len(temporal), "ocorrências")
temporal["pop_titulo"].value_counts().head(10)

### Bairros e perfil

In [ ]:
# Limites dos bairros: a unidade de comparação
bairros = gpd.read_parquet(BQ / "dados_mestres" / "bairro.parquet")
print(bairros.shape)
bairros.head()

In [ ]:
# Censo 2022: população e domicílios por bairro
censo = gpd.read_parquet(DATA_RIO / "censo2022_bairros.parquet")
print(censo.shape)
censo.head()

In [ ]:
# IPS 2022: nota geral e por componente, por bairro
ips = gpd.read_parquet(DATA_RIO / "ips2022_bairros.parquet")
print(ips.shape)
ips.head()

## 4. O que vamos tentar ver

1. **Onde e quando chove**: como a chuva se distribui pela cidade, ao longo do ano e do dia? Os dias extremos ficaram mais frequentes desde 1997?
2. **O que a chuva provoca**: que tipos de ocorrência aumentam em dias de chuva, e onde?
3. **O limiar**: a partir de quantos milímetros a chance de ocorrência passa de 50%? A prefeitura fala em 30 mm/dia.
4. **Pancada ou acumulado**: o que explica melhor cada tipo de ocorrência, chuva curta e forte (15 min, 1 h) ou acumulada (24 h, 96 h)?
5. **Bairro a bairro**: o limiar é igual na cidade toda?
6. **Quem é mais vulnerável**: os bairros que mais sofrem com a mesma chuva têm mais favelas não urbanizadas, mais domicílios inadequados, maior densidade?
7. **A resposta**: com a mesma chuva, o tempo de resposta dos órgãos muda entre regiões?

**Como juntar**: cada ocorrência tem coordenada e horário. Ligamos cada uma ao pluviômetro mais próximo e à chuva das horas anteriores, e depois ao bairro onde caiu.

**Cuidados**
- As bases do datalake vão até 2024 (ocorrências até abril).
- Ocorrência registrada não é o mesmo que alagamento real, e áreas mais monitoradas tendem a registrar mais.
- O campo de bairro das ocorrências está vazio em ~40% dos casos, então o bairro vai sair das coordenadas.